In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

n_customers = 100

# --------------------------
# Customer IDs
# --------------------------

customer_ids = [f"C{i:03d}" for i in range(1, n_customers + 1)]

# --------------------------
# Devices
# Most devices are personal.
# Some are intentionally shared.
# --------------------------

personal_devices = [f"D{i:03d}" for i in range(1, 81)]
shared_devices = ["D_FAMILY_1", "D_FAMILY_2", "D_OFFICE", "D_FRAUD"]

device_pool = (
    personal_devices
    + ["D_FAMILY_1"] * 8
    + ["D_FAMILY_2"] * 6
    + ["D_OFFICE"] * 12
    + ["D_FRAUD"] * 15
)

# --------------------------
# Emails
# Most unique
# Some intentionally shared
# --------------------------

unique_emails = [f"user{i}@example.com" for i in range(1, 81)]

email_pool = (
    unique_emails
    + ["family@gmail.com"] * 6
    + ["shared@yahoo.com"] * 5
    + ["fraud@mailinator.com"] * 8
)

# --------------------------
# Phones
# Some recycled
# --------------------------

phones = [f"+49170000{i:03d}" for i in range(80)]

phone_pool = (
    phones
    + ["+491799999001"] * 4
    + ["+491799999002"] * 4
    + ["+491799999003"] * 6
)

# --------------------------
# IP addresses
# Home IPs + public WiFi + office
# --------------------------

home_ips = [f"192.168.1.{i}" for i in range(1, 61)]

ip_pool = (
    home_ips
    + ["10.0.0.1"] * 10          # Office
    + ["172.16.0.100"] * 12      # Starbucks WiFi
    + ["172.16.0.200"] * 15      # Airport WiFi
)

# --------------------------
# Generate random event dates
# over last 180 days
# --------------------------

dates = pd.to_datetime("2026-01-01") + pd.to_timedelta(
    np.random.randint(0, 180, n_customers),
    unit="D"
)

# --------------------------
# Build dataframe
# --------------------------

df = pd.DataFrame({
    "customer_id": customer_ids,
    "device_id": np.random.choice(device_pool, n_customers),
    "email": np.random.choice(email_pool, n_customers),
    "phone": np.random.choice(phone_pool, n_customers),
    "ip_address": np.random.choice(ip_pool, n_customers),
    "event_time": dates
})

df.head(10)

,customer_id,device_id,email,phone,ip_address,event_time
0,C001,D035,user72@example.com,+49170000018,192.168.1.39,2026-04-13
1,C002,D033,user27@example.com,+49170000047,172.16.0.200,2026-06-29
2,C003,D005,user9@example.com,+49170000079,192.168.1.42,2026-04-03
3,C004,D_OFFICE,user62@example.com,+49170000002,192.168.1.26,2026-01-15
4,C005,D_OFFICE,user37@example.com,+49170000019,192.168.1.50,2026-04-17
5,C006,D041,fraud@mailinator.com,+49170000023,192.168.1.25,2026-03-13
6,C007,D028,user51@example.com,+49170000053,192.168.1.24,2026-01-21
7,C008,D007,user44@example.com,+49170000032,192.168.1.13,2026-04-13
8,C009,D073,user24@example.com,+49170000023,192.168.1.60,2026-05-02
9,C010,D072,user79@example.com,+49170000074,192.168.1.7,2026-03-16


In [ ]:
from grakel import Graph
from grakel.kernels import WeisfeilerLehman, VertexHistogram

print("✓ GraKeL imports successful")

In [2]:
extra_events = df.sample(50, replace=True, random_state=42).copy()

extra_events["event_time"] += pd.to_timedelta(
    np.random.randint(1, 30, len(extra_events)),
    unit="D"
)

df = pd.concat([df, extra_events], ignore_index=True)

df = df.sort_values("event_time").reset_index(drop=True)

print(df.shape)
df.head()

(150, 6)


,customer_id,device_id,email,phone,ip_address,event_time
0,C065,D059,user19@example.com,+49170000029,192.168.1.58,2026-01-02
1,C019,D037,user12@example.com,+491799999001,172.16.0.100,2026-01-02
2,C069,D066,user20@example.com,+49170000036,10.0.0.1,2026-01-04
3,C057,D_FRAUD,user59@example.com,+49170000018,192.168.1.16,2026-01-08
4,C049,D_FRAUD,user69@example.com,+49170000026,10.0.0.1,2026-01-09


Inspect the interesting cases

In [3]:
df.groupby("device_id")["customer_id"].nunique() \
  .sort_values(ascending=False) \
  .head(10)

device_id
D_OFFICE      16
D_FRAUD       11
D_FAMILY_2     6
D062           5
D028           4
D_FAMILY_1     4
D077           2
D044           2
D003           2
D042           2
Name: customer_id, dtype: int64

In [4]:
df.groupby("email")["customer_id"].nunique() \
  .sort_values(ascending=False) \
  .head(10)

email
fraud@mailinator.com    13
shared@yahoo.com         6
user2@example.com        5
user32@example.com       4
user59@example.com       4
family@gmail.com         3
user52@example.com       3
user1@example.com        3
user19@example.com       3
user70@example.com       3
Name: customer_id, dtype: int64

In [5]:
df.groupby("phone")["customer_id"].nunique() \
  .sort_values(ascending=False) \
  .head(10)

phone
+491799999003    6
+49170000032     6
+49170000047     5
+49170000023     4
+491799999002    4
+49170000053     4
+49170000045     3
+49170000066     3
+49170000029     3
+49170000026     3
Name: customer_id, dtype: int64

Build the nodes table

In [6]:
nodes = set()

for row in df.itertuples():

    nodes.add(f"cust_{row.customer_id}")
    nodes.add(f"device_{row.device_id}")
    nodes.add(f"email_{row.email}")
    nodes.add(f"phone_{row.phone}")
    nodes.add(f"ip_{row.ip_address}")

print(len(nodes))

295


In [8]:
for node in list(nodes)[:10]:
    print(node)

device_D030
ip_192.168.1.53
cust_C083
device_D073
cust_C043
email_user20@example.com
email_user52@example.com
cust_C060
phone_+49170000075
ip_192.168.1.19


In [12]:
edges = []

for row in df.itertuples():

    customer = f"cust_{row.customer_id}"

    edges.append(
        (customer,
         f"device_{row.device_id}",
         "uses_device")
    )

    edges.append(
        (customer,
         f"email_{row.email}",
         "has_email")
    )

    edges.append(
        (customer,
         f"phone_{row.phone}",
         "has_phone")
    )

    edges.append(
        (customer,
         f"ip_{row.ip_address}",
         "used_ip")
    )

In [14]:
edges[:10]

[('cust_C065', 'device_D059', 'uses_device'),
 ('cust_C065', 'email_user19@example.com', 'has_email'),
 ('cust_C065', 'phone_+49170000029', 'has_phone'),
 ('cust_C065', 'ip_192.168.1.58', 'used_ip'),
 ('cust_C019', 'device_D037', 'uses_device'),
 ('cust_C019', 'email_user12@example.com', 'has_email'),
 ('cust_C019', 'phone_+491799999001', 'has_phone'),
 ('cust_C019', 'ip_172.16.0.100', 'used_ip'),
 ('cust_C069', 'device_D066', 'uses_device'),
 ('cust_C069', 'email_user20@example.com', 'has_email')]

Build Graph

In [15]:
import networkx as nx

G = nx.Graph()

for source, target, relation in edges:
    G.add_edge(source, target, relation=relation)

In [16]:
G.number_of_nodes()

295

In [17]:
G.number_of_edges()

400

In [18]:
list(G.neighbors("cust_C001"))

['device_D035',
 'email_user72@example.com',
 'phone_+49170000018',
 'ip_192.168.1.39']

Create df

In [19]:
import pandas as pd

node_records = []

for node in nodes:

    node_type, node_value = node.split("_", 1)

    node_records.append({
        "node_id": node,
        "node_type": node_type,
        "node_value": node_value
    })

nodes_df = pd.DataFrame(node_records)

nodes_df.head()

,node_id,node_type,node_value
0,device_D030,device,D030
1,ip_192.168.1.53,ip,192.168.1.53
2,cust_C083,cust,C083
3,device_D073,device,D073
4,cust_C043,cust,C043


In [20]:
edges_df = pd.DataFrame(
    edges,
    columns=[
        "source",
        "target",
        "relationship"
    ]
)

edges_df.head()

,source,target,relationship
0,cust_C065,device_D059,uses_device
1,cust_C065,email_user19@example.com,has_email
2,cust_C065,phone_+49170000029,has_phone
3,cust_C065,ip_192.168.1.58,used_ip
4,cust_C019,device_D037,uses_device


In [21]:
nodes_df["node_type"].value_counts()

node_type
cust      100
phone      53
email      50
device     49
ip         43
Name: count, dtype: int64

Connected Components can become cumbersome. Mitigating strategies

In [22]:
confidence = {
    "has_phone": 0.99,
    "has_email": 0.95,
    "uses_device": 0.75,
    "used_ip": 0.25
}

edges_df["weight"] = edges_df["relationship"].map(confidence)

In [25]:
#add to graph

G = nx.Graph()

for row in edges_df.itertuples():

    G.add_edge(
        row.source,
        row.target,
        relationship=row.relationship,
        weight=row.weight
    )

In [26]:
G["cust_C065"]["email_user19@example.com"]

{'relationship': 'has_email', 'weight': 0.95}

Lecture 2.1

In [29]:
#Node degree
degree_dict = dict(G.degree())

nodes_df["degree"] = (
    nodes_df["node_id"]
        .map(degree_dict)
        .fillna(0)
)


In [30]:
nodes_df.sort_values("degree", ascending=False).head(10)

,node_id,node_type,node_value,degree
39,ip_172.16.0.200,ip,172.16.0.200,18
274,device_D_OFFICE,device,D_OFFICE,16
116,ip_10.0.0.1,ip,10.0.0.1,15
206,email_fraud@mailinator.com,email,fraud@mailinator.com,13
95,device_D_FRAUD,device,D_FRAUD,11
202,email_shared@yahoo.com,email,shared@yahoo.com,6
269,phone_+49170000032,phone,+49170000032,6
142,phone_+491799999003,phone,+491799999003,6
80,device_D_FAMILY_2,device,D_FAMILY_2,6
217,email_user2@example.com,email,user2@example.com,5


In [31]:
weighted_degree = dict(G.degree(weight="weight"))

nodes_df["weighted_degree"] = (
    nodes_df["node_id"]
        .map(weighted_degree)
        .fillna(0)
)

In [32]:
nodes_df.sort_values("weighted_degree", ascending=False).head(10)

,node_id,node_type,node_value,degree,weighted_degree
206,email_fraud@mailinator.com,email,fraud@mailinator.com,13,12.35
274,device_D_OFFICE,device,D_OFFICE,16,12.00
95,device_D_FRAUD,device,D_FRAUD,11,8.25
269,phone_+49170000032,phone,+49170000032,6,5.94
142,phone_+491799999003,phone,+491799999003,6,5.94
202,email_shared@yahoo.com,email,shared@yahoo.com,6,5.70
49,phone_+49170000047,phone,+49170000047,5,4.95
217,email_user2@example.com,email,user2@example.com,5,4.75
39,ip_172.16.0.200,ip,172.16.0.200,18,4.50
80,device_D_FAMILY_2,device,D_FAMILY_2,6,4.50


In [34]:
betweenness = nx.betweenness_centrality(
    G,
    weight="weight"
)

nodes_df["betweenness"] = (
    nodes_df["node_id"]
        .map(betweenness)
)

In [35]:
nodes_df.sort_values("betweenness", ascending=False).head(10)

,node_id,node_type,node_value,degree,weighted_degree,betweenness
39,ip_172.16.0.200,ip,172.16.0.200,18,4.50,0.309228
116,ip_10.0.0.1,ip,10.0.0.1,15,3.75,0.268813
274,device_D_OFFICE,device,D_OFFICE,16,12.00,0.179865
206,email_fraud@mailinator.com,email,fraud@mailinator.com,13,12.35,0.152526
95,device_D_FRAUD,device,D_FRAUD,11,8.25,0.113777
250,cust_C085,cust,C085,4,2.94,0.076786
80,device_D_FAMILY_2,device,D_FAMILY_2,6,4.50,0.076671
73,cust_C022,cust,C022,4,2.94,0.074316
72,device_D_FAMILY_1,device,D_FAMILY_1,4,3.00,0.062492
229,cust_C023,cust,C023,4,2.94,0.059996


In [36]:
closeness = nx.closeness_centrality(G)

nodes_df["closeness"] = (
    nodes_df["node_id"]
        .map(closeness)
)
nodes_df.sort_values("closeness", ascending=False).head(10)

,node_id,node_type,node_value,degree,weighted_degree,betweenness,closeness
39,ip_172.16.0.200,ip,172.16.0.200,18,4.50,0.309228,0.262313
116,ip_10.0.0.1,ip,10.0.0.1,15,3.75,0.268813,0.254329
206,email_fraud@mailinator.com,email,fraud@mailinator.com,13,12.35,0.152526,0.247245
190,cust_C087,cust,C087,4,2.94,0.040623,0.244690
250,cust_C085,cust,C085,4,2.94,0.076786,0.243015
274,device_D_OFFICE,device,D_OFFICE,16,12.00,0.179865,0.242600
20,cust_C027,cust,C027,4,2.94,0.029683,0.240546
76,cust_C051,cust,C051,4,2.94,0.036208,0.236540
254,cust_C049,cust,C049,4,2.94,0.040738,0.236147
174,cust_C079,cust,C079,4,2.94,0.033633,0.236147


In [37]:
eigenvector = nx.eigenvector_centrality(
    G,
    weight="weight",
    max_iter=1000
)

nodes_df["eigenvector"] = (
    nodes_df["node_id"]
        .map(eigenvector)
)
nodes_df.sort_values("eigenvector", ascending=False).head(10)

,node_id,node_type,node_value,degree,weighted_degree,betweenness,closeness,eigenvector
206,email_fraud@mailinator.com,email,fraud@mailinator.com,13,12.35,0.152526,0.247245,0.530219
274,device_D_OFFICE,device,D_OFFICE,16,12.00,0.179865,0.242600,0.297777
242,cust_C028,cust,C028,4,2.94,0.019783,0.222812,0.234127
190,cust_C087,cust,C087,4,2.94,0.040623,0.244690,0.226989
269,phone_+49170000032,phone,+49170000032,6,5.94,0.014973,0.222115,0.197754
174,cust_C079,cust,C079,4,2.94,0.033633,0.236147,0.188744
46,cust_C070,cust,C070,4,2.94,0.014814,0.201051,0.173507
264,cust_C006,cust,C006,4,2.94,0.040504,0.200767,0.172992
48,cust_C100,cust,C100,4,2.94,0.022815,0.234587,0.170473
29,cust_C031,cust,C031,4,2.94,0.029453,0.234975,0.168729


In [38]:
pagerank = nx.pagerank(
    G,
    weight="weight"
)

nodes_df["pagerank"] = (
    nodes_df["node_id"]
        .map(pagerank)
)
nodes_df.sort_values("pagerank", ascending=False).head(10)

,node_id,node_type,node_value,degree,weighted_degree,betweenness,closeness,eigenvector,pagerank
206,email_fraud@mailinator.com,email,fraud@mailinator.com,13,12.35,0.152526,0.247245,5.302189e-01,0.016074
274,device_D_OFFICE,device,D_OFFICE,16,12.00,0.179865,0.242600,2.977772e-01,0.015860
95,device_D_FRAUD,device,D_FRAUD,11,8.25,0.113777,0.225643,1.933667e-02,0.011567
142,phone_+491799999003,phone,+491799999003,6,5.94,0.035002,0.220734,1.104973e-02,0.008656
202,email_shared@yahoo.com,email,shared@yahoo.com,6,5.70,0.034941,0.198800,2.292098e-02,0.008141
44,cust_C010,cust,C010,4,2.94,0.000139,0.013605,1.029525e-16,0.008060
269,phone_+49170000032,phone,+49170000032,6,5.94,0.014973,0.222115,1.977537e-01,0.007715
186,cust_C029,cust,C029,4,2.94,0.019990,0.143695,8.212511e-04,0.007015
70,cust_C073,cust,C073,4,2.94,0.019990,0.145312,8.928658e-05,0.006784
80,device_D_FAMILY_2,device,D_FAMILY_2,6,4.50,0.076671,0.211530,4.133353e-02,0.006781


In [39]:
clustering = nx.clustering(
    G,
    weight="weight"
)

nodes_df["clustering"] = (
    nodes_df["node_id"]
        .map(clustering)
)
nodes_df.sort_values("clustering", ascending=False).head(10)

,node_id,node_type,node_value,degree,weighted_degree,betweenness,closeness,eigenvector,pagerank,clustering
0,device_D030,device,D030,1,0.75,0.000000,0.150868,1.205192e-03,0.001536,0
185,email_user79@example.com,email,user79@example.com,1,0.95,0.000000,0.007775,6.176344e-17,0.002723,0
201,cust_C034,cust,C034,4,2.94,0.024921,0.198522,1.018780e-02,0.004606,0
200,email_user53@example.com,email,user53@example.com,1,0.95,0.000000,0.178110,1.387056e-03,0.001926,0
199,ip_192.168.1.45,ip,192.168.1.45,1,0.25,0.000000,0.166424,9.140469e-03,0.000850,0
198,phone_+49170000031,phone,+49170000031,2,1.98,0.000783,0.201051,3.030326e-03,0.002986,0
197,cust_C007,cust,C007,4,2.94,0.018424,0.177221,3.005192e-03,0.005310,0
196,phone_+49170000008,phone,+49170000008,1,0.99,0.000000,0.171239,2.599986e-03,0.001886,0
195,cust_C025,cust,C025,4,2.94,0.058297,0.218359,5.825680e-03,0.004989,0
194,phone_+49170000058,phone,+49170000058,1,0.99,0.000000,0.178558,2.043413e-03,0.001982,0


In [40]:
component_map = {}

for component in nx.connected_components(G):
    size = len(component)

    for node in component:
        component_map[node] = size

nodes_df["component_size"] = (
    nodes_df["node_id"]
        .map(component_map)
)

nodes_df.sort_values("component_size", ascending=False).head(10)

,node_id,node_type,node_value,degree,weighted_degree,betweenness,closeness,eigenvector,pagerank,clustering,component_size
0,device_D030,device,D030,1,0.75,0.000000,0.150868,0.001205,0.001536,0,290
184,email_user16@example.com,email,user16@example.com,2,1.90,0.000000,0.173965,0.033111,0.003098,0,290
201,cust_C034,cust,C034,4,2.94,0.024921,0.198522,0.010188,0.004606,0,290
200,email_user53@example.com,email,user53@example.com,1,0.95,0.000000,0.178110,0.001387,0.001926,0,290
199,ip_192.168.1.45,ip,192.168.1.45,1,0.25,0.000000,0.166424,0.009140,0.000850,0,290
198,phone_+49170000031,phone,+49170000031,2,1.98,0.000783,0.201051,0.003030,0.002986,0,290
197,cust_C007,cust,C007,4,2.94,0.018424,0.177221,0.003005,0.005310,0,290
196,phone_+49170000008,phone,+49170000008,1,0.99,0.000000,0.171239,0.002600,0.001886,0,290
195,cust_C025,cust,C025,4,2.94,0.058297,0.218359,0.005826,0.004989,0,290
194,phone_+49170000058,phone,+49170000058,1,0.99,0.000000,0.178558,0.002043,0.001982,0,290


Neighborhood variables

In [41]:
from itertools import combinations

# Get all customer nodes
customer_nodes = nodes_df.loc[
    nodes_df["node_type"] == "cust",
    "node_id"
].tolist()

# Example: first 20 customers to keep computation reasonable
sample_customers = customer_nodes[:20]

candidate_pairs = list(combinations(sample_customers, 2))

len(candidate_pairs)

190

In [42]:
shortest_paths = []

for u, v in candidate_pairs:

    try:
        sp = nx.shortest_path_length(G, u, v)
    except nx.NetworkXNoPath:
        sp = None

    shortest_paths.append(sp)

In [44]:
shortest_df = pd.DataFrame({
    "customer_1": [u for u, v in candidate_pairs],
    "customer_2": [v for u, v in candidate_pairs],
    "shortest_path": shortest_paths
})
shortest_df.head()

,customer_1,customer_2,shortest_path
0,cust_C083,cust_C043,6.0
1,cust_C083,cust_C060,4.0
2,cust_C083,cust_C004,6.0
3,cust_C083,cust_C036,4.0
4,cust_C083,cust_C027,4.0


In [45]:
common_neighbors = []

for u, v in candidate_pairs:

    cn = len(list(nx.common_neighbors(G, u, v)))

    common_neighbors.append(cn)

In [46]:
common_neighbors_df = pd.DataFrame({
    "customer_1": [u for u, v in candidate_pairs],
    "customer_2": [v for u, v in candidate_pairs],
    "common_neighbors": common_neighbors
})
common_neighbors_df.head()

,customer_1,customer_2,common_neighbors
0,cust_C083,cust_C043,0
1,cust_C083,cust_C060,0
2,cust_C083,cust_C004,0
3,cust_C083,cust_C036,0
4,cust_C083,cust_C027,0


In [47]:
jaccard = list(
    nx.jaccard_coefficient(
        G,
        candidate_pairs
    )
)

In [48]:
jaccard_df = pd.DataFrame(
    jaccard,
    columns=["source", "target", "jaccard"]
)

In [49]:
jaccard_df.head()

,source,target,jaccard
0,cust_C083,cust_C043,0.0
1,cust_C083,cust_C060,0.0
2,cust_C083,cust_C004,0.0
3,cust_C083,cust_C036,0.0
4,cust_C083,cust_C027,0.0


In [50]:
adamic = list(
    nx.adamic_adar_index(
        G,
        candidate_pairs
    )
)

adamic_df = pd.DataFrame(
    adamic,
    columns=[
        "source",
        "target",
        "adamic_adar"
    ]
)
adamic_df.head()

,source,target,adamic_adar
0,cust_C083,cust_C043,0.0
1,cust_C083,cust_C060,0.0
2,cust_C083,cust_C004,0.0
3,cust_C083,cust_C036,0.0
4,cust_C083,cust_C027,0.0


In [51]:
adamic = list(
    nx.adamic_adar_index(
        G,
        candidate_pairs
    )
)

adamic_df = pd.DataFrame(
    adamic,
    columns=[
        "source",
        "target",
        "adamic_adar"
    ]
)
adamic_df.head()

,source,target,adamic_adar
0,cust_C083,cust_C043,0.0
1,cust_C083,cust_C060,0.0
2,cust_C083,cust_C004,0.0
3,cust_C083,cust_C036,0.0
4,cust_C083,cust_C027,0.0


In [52]:
import numpy as np

# Node order
node_list = list(G.nodes())

A = nx.to_numpy_array(
    G,
    nodelist=node_list
)

beta = 0.05

I = np.eye(A.shape[0])

# Katz similarity matrix
katz = np.linalg.inv(I - beta * A) - I

In [53]:
node_index = {
    node: i
    for i, node in enumerate(node_list)
}

In [54]:
katz_scores = []

for u, v in candidate_pairs:

    i = node_index[u]
    j = node_index[v]

    katz_scores.append(
        katz[i, j]
    )

katz_df = pd.DataFrame(
    list(zip([u for u, v in candidate_pairs], [v for u, v in candidate_pairs], katz_scores)),
    columns=["source", "target", "katz"]
)
katz_df.head()

,source,target,katz
0,cust_C083,cust_C043,2.560439e-09
1,cust_C083,cust_C060,1.120112e-06
2,cust_C083,cust_C004,1.705203e-08
3,cust_C083,cust_C036,3.974948e-07
4,cust_C083,cust_C027,6.076080e-06


In [55]:
pair_features = pd.DataFrame(
    candidate_pairs,
    columns=["customer_1", "customer_2"]
)

pair_features["shortest_path"] = shortest_paths
pair_features["common_neighbors"] = common_neighbors

pair_features = (
    pair_features
        .merge(
            jaccard_df,
            left_on=["customer_1", "customer_2"],
            right_on=["source", "target"],
            how="left"
        )
        .drop(columns=["source", "target"])
)

pair_features = (
    pair_features
        .merge(
            adamic_df,
            left_on=["customer_1", "customer_2"],
            right_on=["source", "target"],
            how="left"
        )
        .drop(columns=["source", "target"])
)

pair_features["katz"] = katz_scores

pair_features.head()

,customer_1,customer_2,shortest_path,common_neighbors,jaccard,adamic_adar,katz
0,cust_C083,cust_C043,6.0,0,0.0,0.0,2.560439e-09
1,cust_C083,cust_C060,4.0,0,0.0,0.0,1.120112e-06
2,cust_C083,cust_C004,6.0,0,0.0,0.0,1.705203e-08
3,cust_C083,cust_C036,4.0,0,0.0,0.0,3.974948e-07
4,cust_C083,cust_C027,4.0,0,0.0,0.0,6.076080e-06
